In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [7]:
class StateAgent(TypedDict):
    num1: int
    num2: int
    num3: int
    num4: int
    op1: str
    op2: str
    mid_result: int
    final_result: int

In [22]:
def adder1(State: StateAgent) -> StateAgent:
    State['mid_result'] = State['num1'] + State['num2']
    return State

def sub1(State: StateAgent) -> StateAgent:
    State['mid_result'] = State['num1']- State['num2']
    return State

def adder2(State: StateAgent) -> StateAgent:
    State['final_result'] = State['num3'] + State['num4']
    return State

def sub2(State: StateAgent) -> StateAgent:
    State['final_result'] = State['num3'] - State['num4']
    return State

def decider1(State: StateAgent) -> StateAgent:
    if State['op1'] == '+':
        return "add_operation1"
    elif State['op1'] == '-':
        return "sub_operation1"
    
def decider2(State: StateAgent) -> StateAgent:
    if State['op2'] == '+':
        return "add_operation2"
    elif State['op2'] == '-':
        return "sub_operation2"

In [23]:
graph = StateGraph(StateAgent)

graph.add_node("router1", lambda State:State)
graph.add_node("adder1", adder1)
graph.add_node("sub1", sub1)

graph.add_node("router2", lambda State:State)
graph.add_node("adder2", adder2)
graph.add_node("sub2", sub2)

graph.add_edge(START, "router1")

graph.add_conditional_edges(
    "router1",
    decider1,
    {
        "add_operation1": "adder1",
        "sub_operation1": "sub1"
    }
)

graph.add_conditional_edges(
    "router2",
    decider2,
    {
        "add_operation2": "adder2",
        "sub_operation2": "sub2"
    }
)

graph.add_edge("adder1", "router2")
graph.add_edge("sub1", "router2")
graph.add_edge("adder2", END)
graph.add_edge("sub2", END)

In [25]:
app = graph.compile()
result = app.invoke({
    "num1": 10,
    "num2": 5,
    "num3": 20,
    "num4": 10,
    "op1": "+",
    "op2": "-"
})

print(result)


{'num1': 10, 'num2': 5, 'num3': 20, 'num4': 10, 'op1': '+', 'op2': '-', 'mid_result': 15, 'final_result': 10}


In [27]:
print(graph)